# Explore anchor and IS alignments

Motivation: issue with false positives arising from fragmented/poorly assembled copies. 
Come up with some alignment quality statistic that allows filtering out hits where the 
IS part only poorly matches the target IS. 

## Set args

In [11]:
class args:
    
    def __init__(self):
        
        self.reads = ['../testing/some_reads.fastq.gz']
        self.target = '../resources/is_targets/IS6110.fasta'
        self.outpath = './'
        self.prefix = 'some_reads'
        self.reference = False
        self.annot = False
        
        self.min_anchor_len = 20
        self.min_hit_len = 20
        self.min_cluster_size = 3
        self.seed_len = 20
        self.tsd_len = [0,3,4]
        self.cpus = 4
        self.keep = False
        self.detailed = True
    
args = args()

In [2]:
import os
import sys
import tempfile

sys.path.append('../')

from lib import readparsing
from lib import clusters

Copy-paste from detettore6110.main()

In [3]:
temp_dir = tempfile.mkdtemp()
target = os.path.abspath(args.target)
reads = readparsing.Reads(args, temp_dir)

# Map reads against IS target ###########################################
readparsing.mapreads(reads.fastq, target, 'reads_vs_IS', temp_dir, 'paf', args.cpus, k=9, m=10)  # Map reads against target IS
reads.parse_paf(f'{temp_dir}/reads_vs_IS.paf', args.min_anchor_len, args.min_hit_len)  # Extract reads that reach into the IS
reads.add_seqs_to_read_dict(reads.fastq, temp_dir)  # Re-traverse reads and extract the anchor sequences

# Identify anchor clusters ##############################################
anchor_clusters = clusters.Clusters(args, temp_dir)
anchor_clusters.cluster_anchors(reads, args.seed_len)  # Cluster anchor sequences based on exact identity of anchor part adjoining IS
anchor_clusters.parse_clusters(reads)  # Align reads and get anchor consensus sequences

In [4]:
anchor_clusters.cluster_d['5'][0].is_seqs

[SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGTTCTTGGAAAGGATGGGGTCATG...CCG'), id='NC_021251.1-1465236/1_0', name='', description='0-81', dbxrefs=[]),
 SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGTTCTTGGAAAGGATGGGGTCAT'), id='NC_021251.1-1189930/1_0', name='', description='0-53', dbxrefs=[]),
 SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGT'), id='NC_021251.1-1127276/1_0', name='', description='0-32', dbxrefs=[]),
 SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGTTCTTGGAAAGGATGGGGTCATG...GCC'), id='NC_021251.1-1077282/1_0', name='', description='0-83', dbxrefs=[]),
 SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGTTCTTGGAAAGGATGGGGTCATG...ACC'), id='NC_021251.1-933260/1_0', name='', description='0-79', dbxrefs=[]),
 SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGTTCTTGGAAAG'), id='NC_021251.1-693526/1_0', name='', description='0-42', dbxrefs=[]),
 SeqRecord(seq=Seq('TGAACCGCCCCGGCATGTCCGGAGACTCCAGTTCTTGGAAAGGATGGGGTCATG...TGG'), id='NC_021251.1-656634/1_0', name='

In [5]:
import subprocess
from Bio import SeqIO

for side in anchor_clusters.cluster_d:
    for cl_id in anchor_clusters.cluster_d[side]:
        
        is_seqs = anchor_clusters.cluster_d[side][cl_id].is_seqs
        
        fasta_path = f'{side}prime{cl_id}.fa'
        aln_path = f'{side}prime{cl_id}.aligned.fa'
        
        with open(fasta_path, 'w') as fasta_handle:
            SeqIO.write(is_seqs, fasta_handle, 'fasta')

        mafft_cmd = ['mafft', '--adjustdirection', fasta_path]

        subprocess.run(mafft_cmd, check=True, stdout=open(aln_path, 'w'), stderr=subprocess.DEVNULL)

In [6]:
temp_dir

'/tmp/tmp0hj72tet'

## Check alignments with MSAviz

In [ ]:
# Import the library
from pymsaviz import MsaViz

# Set parameters and file path
start = 1
end = 50
alignment = ''

# Create visualization object
mv = MsaViz(alignment, wrap_length=100, start=start, end=end, color_scheme="Nucleotide")

# Plot alignment
msa_fig = mv.plotfig()

## Clean up

In [13]:
%%bash
rm *.fa